# Lab 1: Build a CNN from Scratch

Welcome to your first hands-on lab! In this notebook, you will build a Convolutional Neural Network (CNN) from scratch using the **Keras 3 Sequential API** with a **PyTorch backend**.

We will train the model on the classic **MNIST** handwritten digit dataset and evaluate its performance.

## The Vibe-Coding Mindset

Throughout this course, we embrace **vibe coding** -- the practice of describing your intent to an AI assistant (like Claude) and collaborating with it to produce working code. As you work through this lab:

1. **Read each step** to understand what we are trying to accomplish.
2. **Study the code** to see how the intent is translated into implementation.
3. **Experiment** by asking Claude to modify, extend, or explain any section.

The goal is not to memorize syntax -- it is to build intuition for what CNNs do and how to direct an AI to build them for you.

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [1]:
# Set the backend to PyTorch BEFORE importing Keras
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt

print(f"Keras version: {keras.__version__}")
print(f"Backend: {keras.backend.backend()}")

Keras version: 3.13.2
Backend: torch


## Step 1: Load and Explore MNIST

MNIST is a dataset of 70,000 grayscale images of handwritten digits (0-9), each 28x28 pixels. It is the "Hello World" of computer vision.

Let's load it and see what we are working with.

In [ ]:
# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Print the shapes
print(f"Training images shape: {x_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test images shape:     {x_test.shape}")
print(f"Test labels shape:     {y_test.shape}")
print(f"Pixel value range:     [{x_train.min()}, {x_train.max()}]")
print(f"Unique labels:         {np.unique(y_train)}")

# Visualize 10 sample images
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis("off")
plt.suptitle("Sample MNIST Images", fontsize=14)
plt.tight_layout()
plt.show()

## Step 2: Preprocess Data

Before feeding images into a CNN, we need to:

1. **Normalize** pixel values from [0, 255] to [0, 1] so the model trains more efficiently.
2. **Reshape** images to include a channel dimension -- CNNs expect `(height, width, channels)`.
3. **One-hot encode** the labels so the output layer can produce a probability for each class.
4. **Split** a validation set from the training data to monitor overfitting during training.

In [ ]:
# Normalize pixel values to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape to add channel dimension: (28, 28) -> (28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print(f"Reshaped training images: {x_train.shape}")
print(f"Reshaped test images:     {x_test.shape}")

# One-hot encode labels
num_classes = 10
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

print(f"One-hot encoded training labels shape: {y_train.shape}")
print(f"Example one-hot label: {y_train[0]}")

# Split validation set from training data
val_split = 5000
x_val = x_train[:val_split]
y_val = y_train[:val_split]
x_train_split = x_train[val_split:]
y_train_split = y_train[val_split:]

print(f"\nTraining set size:   {x_train_split.shape[0]}")
print(f"Validation set size: {x_val.shape[0]}")
print(f"Test set size:       {x_test.shape[0]}")

## Step 3: Build the CNN Model

We will use the **Keras Sequential API** to stack layers one after another:

- **Conv2D**: Learns spatial filters that detect features like edges, corners, and textures.
- **MaxPooling2D**: Reduces spatial dimensions, making the model more efficient and translation-invariant.
- **Flatten**: Converts the 2D feature maps into a 1D vector for the dense layers.
- **Dense**: Fully connected layers that perform the final classification.

In [ ]:
# Build the CNN model using Sequential API
model = keras.Sequential([
    keras.layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])

# Print the model summary
model.summary()

## Step 4: Compile and Train

Now we configure the training process:

- **Optimizer**: Adam -- an adaptive learning rate optimizer that works well out of the box.
- **Loss**: Categorical crossentropy -- the standard loss for multi-class classification with one-hot labels.
- **Metric**: Accuracy -- the fraction of correctly classified images.

We train for 10 epochs and use a validation split to monitor generalization.

In [ ]:
# Compile the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# Train the model
history = model.fit(
    x_train_split,
    y_train_split,
    epochs=10,
    batch_size=128,
    validation_data=(x_val, y_val),
)

print("\nTraining complete!")

## Step 5: Evaluate and Visualize

Let's see how well the model learned:

1. **Learning curves** -- plot training vs. validation accuracy and loss to check for overfitting.
2. **Test evaluation** -- measure final performance on the held-out test set.
3. **Sample predictions** -- visually inspect some predictions to build intuition.

In [ ]:
# Plot training and validation accuracy and loss
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history["accuracy"], label="Training Accuracy")
ax1.plot(history.history["val_accuracy"], label="Validation Accuracy")
ax1.set_title("Model Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

# Loss
ax2.plot(history.history["loss"], label="Training Loss")
ax2.plot(history.history["val_loss"], label="Validation Loss")
ax2.set_title("Model Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Evaluate on test set
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Show 10 predictions with true vs predicted labels
predictions = model.predict(x_test[:10], verbose=0)
predicted_labels = np.argmax(predictions, axis=1)
true_labels = np.argmax(y_test[:10], axis=1)

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_test[i].squeeze(), cmap="gray")
    color = "green" if predicted_labels[i] == true_labels[i] else "red"
    ax.set_title(f"T:{true_labels[i]} P:{predicted_labels[i]}", color=color, fontsize=10)
    ax.axis("off")
plt.suptitle("Test Predictions (Green = Correct, Red = Wrong)", fontsize=13)
plt.tight_layout()
plt.show()

## Vibe-Coding Reflection

Congratulations -- you have built, trained, and evaluated a CNN from scratch!

Now think about what you would ask Claude to improve. Here are some ideas:

- **"Add dropout layers to reduce overfitting."** -- Dropout randomly disables neurons during training, forcing the network to be more robust.
- **"Add batch normalization after each Conv2D layer."** -- This can speed up training and improve stability.
- **"Show a confusion matrix for the test predictions."** -- A confusion matrix reveals which digits the model confuses most.
- **"Try a learning rate schedule that reduces LR when validation loss plateaus."** -- This can squeeze out extra performance.
- **"Convert this to the Functional API and add a residual connection."** -- A stepping stone toward more advanced architectures.

Pick one and try it! The best way to learn vibe coding is to practice describing what you want and iterating on the result.